# LangGraph Basics with Agentic Assistants

This notebook explores LangGraph integration with the Agentic Assistants framework.

## What is LangGraph?

LangGraph is a library for building stateful, multi-actor applications with LLMs. Key concepts:
- **State**: A shared data structure that flows through the graph
- **Nodes**: Functions that process and transform state
- **Edges**: Connections that define the flow between nodes
- **Conditionals**: Decision points that route to different nodes

## Topics Covered

1. Understanding state graphs
2. Creating nodes with Ollama
3. Building workflows
4. Running with observability
5. Streaming results


In [ ]:
# Setup
from typing import TypedDict, Annotated, Sequence
from operator import add

from agentic_assistants import AgenticConfig, OllamaManager
from agentic_assistants.adapters import LangGraphAdapter
from agentic_assistants.utils.logging import setup_logging

setup_logging(level="INFO")

# Initialize
config = AgenticConfig()
ollama = OllamaManager(config)
adapter = LangGraphAdapter(config)

# Ensure Ollama is ready  
ollama.ensure_running()
model = ollama.ensure_model()
print(f"Using model: {model}")


## Defining State

State is a TypedDict that flows through your graph:


In [ ]:
# Define our workflow state
class QAState(TypedDict):
    """State for a question-answering workflow."""
    question: str           # The user's question
    context: str            # Retrieved or generated context
    answer: str             # The final answer
    confidence: float       # Confidence score
    steps: Annotated[Sequence[str], add]  # Tracks completed steps

print("State schema defined!")


## Creating Nodes

Nodes are functions that transform state:


In [ ]:
# Create LLM instance
llm = adapter.create_ollama_llm()

def analyze_question(state: QAState) -> QAState:
    """Analyze the question to generate context."""
    question = state["question"]
    
    prompt = f"""Analyze this question and provide relevant context:
    
Question: {question}

Provide:
1. Key concepts involved
2. What information is needed to answer
3. Any assumptions to consider"""
    
    response = llm.invoke(prompt)
    
    return {
        "context": response.content,
        "steps": ["question_analyzed"],
    }

def generate_answer(state: QAState) -> QAState:
    """Generate an answer based on context."""
    question = state["question"]
    context = state["context"]
    
    prompt = f"""Based on this analysis, provide a clear answer.

Question: {question}

Context/Analysis: {context}

Provide a direct, helpful answer."""
    
    response = llm.invoke(prompt)
    
    return {
        "answer": response.content,
        "confidence": 0.8,  # Could be computed based on response
        "steps": ["answer_generated"],
    }

def format_response(state: QAState) -> QAState:
    """Format the final response."""
    answer = state["answer"]
    confidence = state["confidence"]
    
    formatted = f"📝 Answer:\n\n{answer}\n\n(Confidence: {confidence:.0%})"
    
    return {
        "answer": formatted,
        "steps": ["response_formatted"],
    }

print("Node functions defined!")


## Building the Graph

Now we assemble the nodes into a workflow:


In [ ]:
from langgraph.graph import StateGraph, END

# Create the graph
graph = adapter.create_state_graph(QAState)

# Add nodes (wrapped for tracing)
graph.add_node("analyze", adapter.wrap_node(analyze_question, "analyze"))
graph.add_node("answer", adapter.wrap_node(generate_answer, "answer"))
graph.add_node("format", adapter.wrap_node(format_response, "format"))

# Define the flow
graph.set_entry_point("analyze")
graph.add_edge("analyze", "answer")
graph.add_edge("answer", "format")
graph.add_edge("format", END)

# Compile the graph
workflow = graph.compile()
print("Workflow compiled!")


## Running the Workflow


In [ ]:
# Initial state
initial_state: QAState = {
    "question": "What are the benefits of using LangGraph for building AI agents?",
    "context": "",
    "answer": "",
    "confidence": 0.0,
    "steps": [],
}

# Run with tracking
result = adapter.run_graph(
    workflow,
    inputs=initial_state,
    experiment_name="langgraph-notebook",
    run_name="qa-workflow-demo",
)

print("Steps completed:", result["steps"])
print("\n" + "="*60)
print(result["answer"])


## Streaming Execution

You can also stream results as they're generated:


In [ ]:
# Streaming example
initial_state["question"] = "How does observability help in AI development?"
initial_state["steps"] = []

print("Streaming execution:")
print("-" * 40)

for step_output in adapter.stream_graph(
    workflow,
    inputs=initial_state,
    experiment_name="langgraph-streaming",
):
    for node_name, node_output in step_output.items():
        print(f"✓ Node: {node_name}")
        if "steps" in node_output:
            print(f"  Steps: {node_output['steps']}")


## What's Being Tracked?

The LangGraph adapter provides:

**MLFlow:**
- Workflow parameters
- Total execution duration
- Success/failure status
- Final state as artifact

**OpenTelemetry:**
- Span for entire workflow
- Child spans for each node
- Node execution times
- State transitions

View traces in Jaeger UI: http://localhost:16686
